# 2 - Geometric Processing (Resize & Padding)

This notebook reads raw, unresized PNG crops from Dataset/1_Raw_Crops/, applies the geometric pipeline (background color estimation, aspect-preserving resize, canvas padding) and writes processed images to Dataset/2_Processed_Padded/.

Progress is shown with `tqdm`. This notebook does NOT import YOLO or PDF libraries.

In [ ]:
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

# PROJECT_ROOT and dataset folders (adjust PROJECT_ROOT if needed)
import platform
OS_TYPE = platform.system()
if OS_TYPE == 'Linux':
    PROJECT_ROOT = Path('/home/vasco/Tese_Vasco_Lnx')
elif OS_TYPE == 'Windows':
    PROJECT_ROOT = Path(r'C:/path/to/project')
else:
    PROJECT_ROOT = Path.home() / 'Tese_Vasco_Lnx'

RAW_DIR = PROJECT_ROOT / 'Dataset' / '1_Raw_Crops'
OUT_DIR = PROJECT_ROOT / 'Dataset' / '2_Processed_Padded'
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Raw crops:', RAW_DIR)
print('Processed out:', OUT_DIR)


In [ ]:
def calculate_background_color(image, inset_px=8):
    img = image.convert('RGB')
    arr = np.asarray(img)
    h, w = arr.shape[:2]
    if h == 0 or w == 0:
        return (255, 255, 255)
    border = max(1, min(int(inset_px), max(1, min(h, w) // 10)))
    top = arr[:border, :]
    bottom = arr[-border:, :]
    left = arr[:, :border]
    right = arr[:, -border:]
    bands = [band.reshape(-1, 3) for band in [top, bottom, left, right] if band.size > 0]
    if not bands:
        return (255, 255, 255)
    border_pixels = np.concatenate(bands, axis=0)
    if border_pixels.size == 0:
        return (255, 255, 255)
    med_color = np.median(border_pixels, axis=0)
    luminance = 0.299 * med_color[0] + 0.587 * med_color[1] + 0.114 * med_color[2]
    if luminance > 200:
        return (255, 255, 255)
    if luminance < 30:
        return (0, 0, 0)
    return tuple(int(np.clip(c, 0, 255)) for c in med_color)

def resize_maintaining_aspect(image, size=256):
    if not isinstance(image, Image.Image):
        raise TypeError('image must be a PIL.Image.Image instance')
    target = int(size)
    img = image.convert('RGB')
    src_w, src_h = img.size
    scale = min(target / src_w, target / src_h)
    new_w = max(1, int(round(src_w * scale)))
    new_h = max(1, int(round(src_h * scale)))
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)

def apply_canvas_padding(image, size=256, fill_color=(0,0,0)):
    if not isinstance(image, Image.Image):
        raise TypeError('image must be a PIL.Image.Image instance')
    target = int(size)
    canvas = Image.new('RGB', (target, target), fill_color)
    offset_x = (target - image.size[0]) // 2
    offset_y = (target - image.size[1]) // 2
    canvas.paste(image, (offset_x, offset_y))
    return canvas

def process_image_pipeline(image, size=256):
    background_color = calculate_background_color(image)
    resized_image = resize_maintaining_aspect(image, size=size)
    return apply_canvas_padding(resized_image, size=size, fill_color=background_color)


In [ ]:
# Iterate through RAW_DIR and process images with a tqdm progress bar
from pathlib import Path
files = list(RAW_DIR.rglob('*.png'))
print(f'Found {len(files)} raw images')
for p in tqdm(files, desc='Processing images'):
    try:
        img = Image.open(p)
        processed = process_image_pipeline(img, size=256)
        out_path = OUT_DIR / p.name
        processed.save(out_path)
    except Exception as e:
        print('Failed:', p, e)

print('Processing complete')
